# Part 1 - Steel Production Route Emission Intensities

## Purpose
Construct a reference table of global average CO2 and energy emission
intensities by steel production route, sourced from the Worldsteel
Sustainability Indicators report (2025 edition, covering 2022-2024 data).

This table serves as a benchmark reference in the analysis layer:
given a country's CBAM default emission value, how does it compare
to the global average for each production route?

## Source
Worldsteel Sustainability Indicators Report 2025

![WorlSteel Emission Intesities by Production Route](assets/worldsteel_emission_intensity.png)

URL: https://worldsteel.org/wider-sustainability/sustainability-indicators/
License: Worldsteel open publication, cited with attribution

## Output
`data/processed/steel_route_intensity.csv`

## Notes
- Data manually constructed from the published table on the Worldsteel
  website, cross-checked against IRENA and World Steel in Figures 2025.
- Global averages only, no country-level breakdown available publicly.
- DRI-EAF denominator estimated by Worldsteel from collective databases
  as global DRI production data is not directly collected.
- Units: tCO2 per tonne of crude steel (emissions), GJ per tonne (energy)

In [ ]:
import pandas as pd
from pathlib import Path

# Source: Worldsteel Sustainability Indicators 2025
# https://worldsteel.org/wider-sustainability/sustainability-indicators/
# Table: CO2 emissions and energy intensity, 2022-2024 (added February 2026)

steel_routes = pd.DataFrame([
    {"production_route": "BF-BOF",     "year": 2022, "co2_intensity_tco2_per_t": 2.33, "energy_intensity_gj_per_t": 23.98},
    {"production_route": "BF-BOF",     "year": 2023, "co2_intensity_tco2_per_t": 2.33, "energy_intensity_gj_per_t": 24.24},
    {"production_route": "BF-BOF",     "year": 2024, "co2_intensity_tco2_per_t": 2.34, "energy_intensity_gj_per_t": 23.88},
    {"production_route": "Scrap-EAF",  "year": 2022, "co2_intensity_tco2_per_t": 0.67, "energy_intensity_gj_per_t": 10.13},
    {"production_route": "Scrap-EAF",  "year": 2023, "co2_intensity_tco2_per_t": 0.69, "energy_intensity_gj_per_t": 10.21},
    {"production_route": "Scrap-EAF",  "year": 2024, "co2_intensity_tco2_per_t": 0.69, "energy_intensity_gj_per_t":  9.84},
    {"production_route": "DRI-EAF",    "year": 2022, "co2_intensity_tco2_per_t": 1.36, "energy_intensity_gj_per_t": 22.25},
    {"production_route": "DRI-EAF",    "year": 2023, "co2_intensity_tco2_per_t": 1.43, "energy_intensity_gj_per_t": 23.13},
    {"production_route": "DRI-EAF",    "year": 2024, "co2_intensity_tco2_per_t": 1.47, "energy_intensity_gj_per_t": 23.30},
    {"production_route": "Global avg", "year": 2022, "co2_intensity_tco2_per_t": 1.92, "energy_intensity_gj_per_t": 21.01},
    {"production_route": "Global avg", "year": 2023, "co2_intensity_tco2_per_t": 1.92, "energy_intensity_gj_per_t": 21.30},
    {"production_route": "Global avg", "year": 2024, "co2_intensity_tco2_per_t": 1.92, "energy_intensity_gj_per_t": 20.95},
])

output_path = Path.cwd().parent / "data" / "processed" / "steel_route_intensity.csv"
steel_routes.to_csv(output_path, index=False)

print(steel_routes.to_string(index=False))
print(f"\nSaved to: {output_path}")
print(f"Shape: {steel_routes.shape}")

production_route  year  co2_intensity_tco2_per_t  energy_intensity_gj_per_t
          BF-BOF  2022                      2.33                      23.98
          BF-BOF  2023                      2.33                      24.24
          BF-BOF  2024                      2.34                      23.88
       Scrap-EAF  2022                      0.67                      10.13
       Scrap-EAF  2023                      0.69                      10.21
       Scrap-EAF  2024                      0.69                       9.84
         DRI-EAF  2022                      1.36                      22.25
         DRI-EAF  2023                      1.43                      23.13
         DRI-EAF  2024                      1.47                      23.30
      Global avg  2022                      1.92                      21.01
      Global avg  2023                      1.92                      21.30
      Global avg  2024                      1.92                      20.95

Saved to: /

---

# Part 2 — Steel Production Route Mix by Country

## Purpose
Construct a reference table of crude steel production volumes and process route shares
(BOF % vs EAF %) by country and regional aggregate from the Worldsteel
World Steel in Figures 2025 PDF.

This table is used in the analysis layer to contextualize a country's
CBAM exposure: whether it primarily uses the high-emission BF-BOF route
or the lower-emission EAF route, and what grid decarbonization would mean
for its CBAM bill under an EAF scenario.

## Source
World Steel in Figures 2025, Worldsteel Association
Table: 'Crude steel production by process, 2024' (PDF page 6)
PDF: `data/raw/World-Steel-in-Figures-2025.pdf`
URL: https://worldsteel.org/data/world-steel-in-figures/world-steel-in-figures-2025/
License: Worldsteel open publication, cited with attribution

## Output
`data/processed/steel_route_mix.csv`

## Notes
- Extracted via pdfplumber. Table parses cleanly with no manual fallback required.
- The source table has three process columns: Oxygen (BOF), Electric (EAF),
  and Other (open hearth furnace, OHF). OHF is an older, more emissions-
  intensive primary route found mainly in Ukraine (38.2%) and Russia (2.2%).
  OHF is rolled into `bof_pct_combined` since both are primary steelmaking
  routes with broadly similar emissions profiles relative to EAF.
  The original `ohf_pct` column is retained for transparency.
- `is_regional_aggregate` flags rows representing regional averages rather
  than individual countries, for easy filtering downstream.
- `is_estimate` flags rows marked with 'e' in the source.
- Country-to-region mapping for CBAM countries not individually listed
  is handled in notebook 07 using the country_crosswalk.
- Units: million tonnes (crude steel production), % (route shares)

In [1]:
# ── Extract table from PDF ────────────────────────────────────────────────────
# 'Crude steel production by process, 2024' is on PDF page 6 (0-indexed: page 5).
# pdfplumber extracts it as a clean 59-row x 6-col table.
# Table 0 on this page is the process mix table; Table 1 is continuously-cast
# steel output, a different dataset not extracted here.

import pdfplumber
import pandas as pd
from pathlib import Path

pdf_path = Path.cwd().parent / 'data' / 'raw' / 'World-Steel-in-Figures-2025.pdf'
assert pdf_path.exists(), f'PDF not found at {pdf_path}'

with pdfplumber.open(pdf_path) as pdf:
    page  = pdf.pages[5]
    table = page.extract_tables()[0]

print(f'Raw rows extracted: {len(table)}')
print(f'Columns: {table[0]}')
print('\nFirst 5 data rows:')
for row in table[1:6]:
    print(f'  {row}')

Raw rows extracted: 59
Columns: ['', 'Million\ntonnes', 'Oxygen\n%', 'Electric\n%', 'Other\n%', 'Total\n%']

First 5 data rows:
  ['Austria', '7.1', '91.6', '8.4', '-', '100.0']
  ['Belgium e', '7.1', '71.2', '28.8', '-', '100.0']
  ['Bulgaria', '0.5', '-', '100.0', '-', '100.0']
  ['Croatia', '0.2', '-', '100.0', '-', '100.0']
  ['Czechia', '2.5', '94.7', '5.3', '-', '100.0']


In [4]:
# ── Parse and clean extracted table ──────────────────────────────────────────
# Steps:
#   1. Load into DataFrame with clean column names
#   2. Strip estimate marker ' e' from country names, flag in is_estimate
#   3. Replace '-' (zero in source) with 0.0 in numeric columns
#   4. Strip thousand-separator spaces (e.g. '1 005.1' -> '1005.1')
#   5. Cast numeric columns to float
#   6. Drop total_pct column (always 100.0, no analytical value)
#   7. Flag regional aggregate rows
#   8. Drop the grand total row

REGIONAL_AGGREGATES = {
    'European Union (27)', 'Other Europe',
    'Russia & other CIS + Ukraine', 'Other CIS',
    'North America', 'Other North America',
    'South America', 'Other South America',
    'Africa', 'Other Africa',
    'Middle East', 'Other Middle East',
    'Asia', 'Other Asia', 'Others',
}

header = ['country', 'production_mt', 'bof_pct', 'eaf_pct', 'ohf_pct', 'total_pct']
df = pd.DataFrame(table[1:], columns=header)

df['is_estimate'] = df['country'].str.endswith(' e')
df['country']     = df['country'].str.replace(r' e$', '', regex=True).str.strip()

numeric_cols = ['production_mt', 'bof_pct', 'eaf_pct', 'ohf_pct', 'total_pct']
for col in numeric_cols:
    df[col] = (
        df[col]
        .str.replace(' ', '', regex=False)
        .replace('-', '0.0')
        .astype(float)
    )

df = df.drop(columns='total_pct')
df['is_regional_aggregate'] = df['country'].isin(REGIONAL_AGGREGATES)
df = df[df['country'] != 'Total of above countries'].reset_index(drop=True)

# OHF rolled into BOF for analysis. Original ohf_pct retained for transparency.
df['bof_pct_combined'] = df['bof_pct'] + df['ohf_pct']

print(f'Rows after cleaning:   {len(df)}')
print(f'Individual countries:  {(~df["is_regional_aggregate"]).sum()}')
print(f'Regional aggregates:   {df["is_regional_aggregate"].sum()}')
print(f'Rows flagged estimate: {df["is_estimate"].sum()}')
print('\nRows with OHF production:')
print(df[df['ohf_pct'] > 0][['country', 'bof_pct', 'eaf_pct', 'ohf_pct', 'bof_pct_combined']].to_string(index=False))

Rows after cleaning:   57
Individual countries:  42
Regional aggregates:   15
Rows flagged estimate: 16

Rows with OHF production:
                     country  bof_pct  eaf_pct  ohf_pct  bof_pct_combined
                      Russia     63.6     34.2      2.2              65.8
                     Ukraine     49.5     12.3     38.2              87.7
                   Other CIS     48.5     49.9      1.7              50.2
Russia & other CIS + Ukraine     60.9     33.8      5.3              66.2
                      Brazil     75.5     23.2      1.2              76.7
               South America     66.6     32.4      1.0              67.6
                  Other Asia     52.5     41.2      6.4              58.9
                        Asia     80.6     19.1      0.3              80.9


In [5]:
# ── Sanity checks and save ────────────────────────────────────────────────────

totals = (df['bof_pct'] + df['eaf_pct'] + df['ohf_pct']).round(1)
assert (totals.between(99.9, 100.1)).all(), \
    f'Rows not summing to 100%:\n{df[totals.between(99.9, 100.1) == False][["country", "bof_pct", "eaf_pct", "ohf_pct"]]}'
print('PASS: All rows sum to 100%')

assert (df['bof_pct_combined'] >= df['bof_pct']).all(), \
    'bof_pct_combined less than bof_pct for some rows'
print('PASS: bof_pct_combined >= bof_pct for all rows')

assert df.isnull().sum().sum() == 0, f'Nulls found:\n{df.isnull().sum()}'
print('PASS: No nulls')

output_path = Path.cwd().parent / 'data' / 'processed' / 'steel_route_mix.csv'
df.to_csv(output_path, index=False)

print(f'\nSaved to: {output_path}')
print(f'Shape: {df.shape}')
print(df.to_string(index=False))

PASS: All rows sum to 100%
PASS: bof_pct_combined >= bof_pct for all rows
PASS: No nulls

Saved to: /Users/milcahmaryjoseph/Documents/GitHub/cbam-analysis/data/processed/steel_route_mix.csv
Shape: (57, 8)
                     country  production_mt  bof_pct  eaf_pct  ohf_pct  is_estimate  is_regional_aggregate  bof_pct_combined
                     Austria            7.1     91.6      8.4      0.0        False                  False              91.6
                     Belgium            7.1     71.2     28.8      0.0         True                  False              71.2
                    Bulgaria            0.5      0.0    100.0      0.0        False                  False               0.0
                     Croatia            0.2      0.0    100.0      0.0        False                  False               0.0
                     Czechia            2.5     94.7      5.3      0.0        False                  False              94.7
                     Finland            3.7  